# Genome Selection and Representative Sampling

Este notebook realiza el análisis completo de genomas de *H. pylori*, incluyendo:
- Descarga y organización de datos de referencia mundial
- Descarga de cepas peruanas desde NCBI
- Análisis de ortólogos con OrthoFinder
- Análisis de diversidad y diferenciación genética
- Visualizaciones de resultados

## Importación de Librerías

In [ ]:
# Librerías del sistema
import os
import sys
import time
import glob
import shutil
import zipfile
import tarfile
import subprocess
from pathlib import Path
from collections import defaultdict, Counter

# Librerías de datos y análisis
import pandas as pd
import numpy as np
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# Librerías de visualización
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

# Librerías de descarga
import requests
import gdown

## 1. Descarga y Organización de Archivos

### 1.1 Configuración de Rutas y Constantes

In [ ]:
# IDs de Google Drive
ID_EXCEL = '16tt8sjX2oN3H6I0_7VLtyvsT6Jj4oJjC'
ID_ZIP_REFS = '1bBG-cwhb3u-ydIo_A1LoO04yxkOrLhNI'

# Rutas de trabajo
RUTA_REFS_PROT = "/content/Proteomas_referencia"
RUTA_REFS_GEN = "/content/Genomas_referencia"
RUTA_ESTUDIO_PROT = "/content/Proteomas_estudio"
RUTA_ESTUDIO_GEN = "/content/Genomas_estudio"
RUTA_EXCEL = "/content/Supplementary_Data_2_255_refs.xlsx"

# Accesiones de cepas peruanas
ACCESIONES_PERU = [
    "GCA_016749025.1", "GCA_016748925.1", "GCA_016748795.1", "GCA_016748675.1",
    "GCA_016748555.1", "GCA_016749715.1", "GCA_016749815.1", "GCA_016749575.1",
    "GCA_016749455.1", "GCA_016749215.1", "GCA_016749335.1", "GCA_016749085.1",
    "GCA_016755815.1", "GCA_016755795.1", "GCA_016755775.1", "GCA_016755755.1",
    "GCA_016755735.1", "GCA_016755715.1", "GCA_016755695.1", "GCA_016755675.1",
    "GCA_016755655.1", "GCA_016755635.1", "GCA_016755615.1", "GCA_016757595.1",
    "GCA_016757535.1", "GCA_016757575.1", "GCA_016757555.1"
]

### 1.2 Funciones de Descarga y Organización

In [ ]:
def descargar_metadatos_referencia():
    """Descarga el archivo Excel con metadatos desde Drive."""
    print("Descargando metadatos (Excel) desde Drive...")
    url_excel = f'https://drive.google.com/uc?id={ID_EXCEL}'
    gdown.download(url_excel, RUTA_EXCEL, quiet=True)

def validar_metadatos_referencia():
    """Carga y valida el archivo Excel de metadatos."""
    if not os.path.exists(RUTA_EXCEL):
        print(f"Error: No se encontró el archivo Excel en {RUTA_EXCEL}")
        return None

    df_refs = pd.read_excel(RUTA_EXCEL, skiprows=0)
    df_refs.columns = df_refs.columns.astype(str).str.strip().str.replace('\n', ' ')

    col_pop = 'Previously assigned Hp population'
    if col_pop not in df_refs.columns:
        print(f"Error: No encontré la columna '{col_pop}'")
        return None
    
    print(f"Éxito: Se cargaron los metadatos de {len(df_refs)} genomas.")
    return df_refs

def descargar_y_clasificar_referencias():
    """Descarga y organiza las secuencias de referencia mundial."""
    os.makedirs(RUTA_REFS_PROT, exist_ok=True)
    os.makedirs(RUTA_REFS_GEN, exist_ok=True)
    ruta_tmp = "/content/temp_refs"
    os.makedirs(ruta_tmp, exist_ok=True)

    url_archivo = f'https://drive.google.com/uc?id={ID_ZIP_REFS}'
    output_archivo = "/content/referencias.tar.gz"

    print("\nDescargando secuencias de referencia mundiales desde Drive...")
    gdown.download(url_archivo, output_archivo, quiet=True)

    print("Extrayendo archivos (tar.gz)...")
    try:
        with tarfile.open(output_archivo, "r:gz") as tar_ref:
            tar_ref.extractall(path=ruta_tmp, filter='data')
    except tarfile.ReadError:
        print("Error: El archivo no es un .tar.gz válido.")
        return

    print("Clasificando archivos...")
    for root, dirs, files in os.walk(ruta_tmp):
        for fname in files:
            ruta_origen = os.path.join(root, fname)
            if fname.endswith(".faa"):
                shutil.move(ruta_origen, os.path.join(RUTA_REFS_PROT, fname))
            elif fname.endswith(".ffn"):
                shutil.move(ruta_origen, os.path.join(RUTA_REFS_GEN, fname))

    shutil.rmtree(ruta_tmp)
    n_prot = len(os.listdir(RUTA_REFS_PROT))
    n_gen = len(os.listdir(RUTA_REFS_GEN))
    print(f"Listo: {n_prot} proteomas y {n_gen} genomas/CDS de referencia organizados.")

def descargar_cepas_ncbi():
    """Descarga cepas peruanas desde NCBI."""
    os.makedirs(RUTA_ESTUDIO_PROT, exist_ok=True)
    os.makedirs(RUTA_ESTUDIO_GEN, exist_ok=True)

    print(f"\nIniciando descarga de {len(ACCESIONES_PERU)} cepas peruanas desde NCBI...")
    
    for acc in ACCESIONES_PERU:
        prefijo = acc.replace('.', '_')
        prot_file = os.path.join(RUTA_ESTUDIO_PROT, f"Peru_{acc}.faa")
        cds_file = os.path.join(RUTA_ESTUDIO_GEN, f"Peru_{acc}.ffn")

        if os.path.exists(prot_file) and os.path.exists(cds_file):
            continue

        url_prot = f"https://ftp.ncbi.nlm.nih.gov/genomes/all/{acc[:3]}/{acc[4:7]}/{acc[7:10]}/{acc[10:13]}/{acc}/"
        
        try:
            resp = requests.get(url_prot, timeout=10)
            if resp.status_code != 200:
                raise Exception("No se encontró GenBank")
            
            lines = resp.text.split('\n')
            basename = None
            for line in lines:
                if '_protein.faa.gz' in line:
                    parts = line.split('"')
                    for p in parts:
                        if p.endswith('_protein.faa.gz'):
                            basename = p.replace('_protein.faa.gz', '')
                            break
                if basename:
                    break
            
            if not basename:
                raise Exception("No basename")

            prot_gz = basename + "_protein.faa.gz"
            cds_gz = basename + "_cds_from_genomic.fna.gz"

            r_prot = requests.get(url_prot + prot_gz, timeout=30)
            r_cds = requests.get(url_prot + cds_gz, timeout=30)

            if r_prot.status_code == 200 and r_cds.status_code == 200:
                with open(prot_file, 'wb') as f:
                    subprocess.run(['gunzip'], input=r_prot.content, stdout=f, check=True)
                with open(cds_file, 'wb') as f:
                    subprocess.run(['gunzip'], input=r_cds.content, stdout=f, check=True)
                print(f" -> ÉXITO: Guardado como Peru_{acc}")
            else:
                raise Exception("Archivos no encontrados")

        except Exception as e:
            acc_refseq = acc.replace('GCA_', 'GCF_')
            url_refseq = f"https://ftp.ncbi.nlm.nih.gov/genomes/all/{acc_refseq[:3]}/{acc_refseq[4:7]}/{acc_refseq[7:10]}/{acc_refseq[10:13]}/{acc_refseq}/"
            
            print(f"    ... Reintentando con RefSeq: {acc_refseq}")
            try:
                resp_r = requests.get(url_refseq, timeout=10)
                if resp_r.status_code != 200:
                    raise Exception("RefSeq no encontrado")
                
                lines_r = resp_r.text.split('\n')
                basename_r = None
                for line in lines_r:
                    if '_protein.faa.gz' in line:
                        parts = line.split('"')
                        for p in parts:
                            if p.endswith('_protein.faa.gz'):
                                basename_r = p.replace('_protein.faa.gz', '')
                                break
                    if basename_r:
                        break
                
                if not basename_r:
                    raise Exception("No basename RefSeq")

                prot_gz_r = basename_r + "_protein.faa.gz"
                cds_gz_r = basename_r + "_cds_from_genomic.fna.gz"

                r_prot_r = requests.get(url_refseq + prot_gz_r, timeout=30)
                r_cds_r = requests.get(url_refseq + cds_gz_r, timeout=30)

                if r_prot_r.status_code == 200 and r_cds_r.status_code == 200:
                    with open(prot_file, 'wb') as f:
                        subprocess.run(['gunzip'], input=r_prot_r.content, stdout=f, check=True)
                    with open(cds_file, 'wb') as f:
                        subprocess.run(['gunzip'], input=r_cds_r.content, stdout=f, check=True)
                    print(f" -> ÉXITO: Guardado como Peru_{acc}")
                else:
                    print(f" -> FALLO: {acc} (RefSeq tampoco funcionó)")
            except Exception:
                print(f" -> FALLO: {acc} (no disponible)")
        
        time.sleep(0.5)

    n_prot_estudio = len(os.listdir(RUTA_ESTUDIO_PROT))
    n_cds_estudio = len(os.listdir(RUTA_ESTUDIO_GEN))
    print(f"\nRESUMEN: {n_prot_estudio} proteomas y {n_cds_estudio} CDS peruanos listos.")

def limpiar_archivos_temporales():
    """Elimina archivos comprimidos temporales."""
    print("\nRealizando limpieza de archivos comprimidos...")
    for item in os.listdir('/content'):
        ruta = os.path.join('/content', item)
        if item.endswith(('.tar.gz', '.zip')) or item == 'temp_refs':
            if os.path.isfile(ruta):
                os.remove(ruta)
            elif os.path.isdir(ruta):
                shutil.rmtree(ruta)
    print("Limpieza completada.")

### 1.3 Ejecución del Pipeline de Descarga

In [ ]:
print("=== INICIANDO PIPELINE DE DESCARGA Y CLASIFICACIÓN ===")

descargar_metadatos_referencia()
df_metadata = validar_metadatos_referencia()

descargar_y_clasificar_referencias()
descargar_cepas_ncbi()
limpiar_archivos_temporales()

print("\n=== PIPELINE FINALIZADO CON ÉXITO ===")

## 2. Análisis con OrthoFinder

### 2.1 Instalación de OrthoFinder

In [ ]:
!wget -q https://github.com/davidemms/OrthoFinder/releases/download/2.5.5/OrthoFinder_source.tar.gz
!tar -xzf OrthoFinder_source.tar.gz
!pip install -q biopython scipy numpy matplotlib

### 2.2 Preparación de Datos

In [ ]:
RUTA_INPUT = "/content/OrthoFinder_Input"
os.makedirs(RUTA_INPUT, exist_ok=True)

for f in os.listdir(RUTA_REFS_PROT):
    if f.endswith(".faa"):
        shutil.copy(os.path.join(RUTA_REFS_PROT, f), RUTA_INPUT)

for f in os.listdir(RUTA_ESTUDIO_PROT):
    if f.endswith(".faa"):
        shutil.copy(os.path.join(RUTA_ESTUDIO_PROT, f), RUTA_INPUT)

print(f"Total de archivos preparados: {len(os.listdir(RUTA_INPUT))}")

### 2.3 Ejecución de OrthoFinder

In [ ]:
!python /content/OrthoFinder/orthofinder.py -f {RUTA_INPUT} -t 2 -a 2

### 2.4 Análisis de Resultados de OrthoFinder

In [ ]:
resultados = glob.glob("/content/OrthoFinder_Input/OrthoFinder/Results_*/Orthogroups/Orthogroups.tsv")

if not resultados:
    print("No se encontraron resultados de OrthoFinder")
else:
    archivo_orthogroups = resultados[0]
    df_ortho = pd.read_csv(archivo_orthogroups, sep='\t')
    print(f"Total de orthogroups identificados: {len(df_ortho)}")
    print(f"Columnas: {df_ortho.columns.tolist()}")

## 3. Filtrado de Genes de Virulencia

### 3.1 Descarga de Archivos Anotados

In [ ]:
ID_ANOTADOS = '1Ot4Dt4G-RU7yEV--gkKA4LaBLk8w6bev'
url_anotados = f'https://drive.google.com/uc?id={ID_ANOTADOS}'
output_anotados = '/content/genomas_anotados.zip'

print("Descargando genomas anotados desde Drive...")
gdown.download(url_anotados, output_anotados, quiet=True)

ruta_anotados = '/content/genomas_anotados'
os.makedirs(ruta_anotados, exist_ok=True)

with zipfile.ZipFile(output_anotados, 'r') as zip_ref:
    zip_ref.extractall(ruta_anotados)

archivos_gff = glob.glob(f"{ruta_anotados}/**/*.gff", recursive=True)
print(f"Archivos GFF encontrados: {len(archivos_gff)}")

### 3.2 Extracción de Genes de Virulencia

In [ ]:
GENES_VIRULENCIA = [
    "cagA", "vacA", "babA", "oipA", "iceA", "sabA", "dupA",
    "napA", "ureA", "ureB", "hpaA", "hopQ", "homA", "homB"
]

def extraer_genes_virulencia_gff(archivo_gff, genes_interes):
    """Extrae información de genes de virulencia desde archivos GFF."""
    genes_encontrados = []
    
    with open(archivo_gff, 'r') as f:
        for linea in f:
            if linea.startswith('#'):
                continue
            
            campos = linea.strip().split('\t')
            if len(campos) < 9:
                continue
            
            tipo = campos[2]
            if tipo not in ['gene', 'CDS']:
                continue
            
            atributos = campos[8]
            
            for gen in genes_interes:
                if gen.lower() in atributos.lower():
                    genes_encontrados.append({
                        'gen': gen,
                        'tipo': tipo,
                        'inicio': campos[3],
                        'fin': campos[4],
                        'atributos': atributos
                    })
    
    return genes_encontrados

resultados_virulencia = {}

for gff_file in archivos_gff:
    nombre_genoma = os.path.basename(gff_file).replace('.gff', '')
    genes = extraer_genes_virulencia_gff(gff_file, GENES_VIRULENCIA)
    resultados_virulencia[nombre_genoma] = genes

total_genes = sum(len(v) for v in resultados_virulencia.values())
print(f"Total de genes de virulencia identificados: {total_genes}")

### 3.3 Filtrado de Orthogroups por Genes de Virulencia

In [ ]:
def filtrar_orthogroups_virulencia(df_ortho, genes_virulencia):
    """Filtra orthogroups que contienen genes de virulencia."""
    patron = '|'.join([f'(?i){gen}' for gen in genes_virulencia])
    
    mask = df_ortho.iloc[:, 1:].apply(
        lambda col: col.astype(str).str.contains(patron, regex=True, na=False)
    ).any(axis=1)
    
    df_filtrado = df_ortho[mask].copy()
    return df_filtrado

if 'df_ortho' in locals():
    df_virulencia = filtrar_orthogroups_virulencia(df_ortho, GENES_VIRULENCIA)
    print(f"Orthogroups con genes de virulencia: {len(df_virulencia)}")
    print(f"\nPrimeros orthogroups identificados:")
    print(df_virulencia.head())

## 4. Análisis de Diversidad Genética

### 4.1 Instalación de GitHub CLI

In [ ]:
!type -p curl >/dev/null || (apt update && apt install curl -y)
!curl -fsSL https://cli.github.com/packages/githubcli-archive-keyring.gpg | dd of=/usr/share/keyrings/githubcli-archive-keyring.gpg
!chmod go+r /usr/share/keyrings/githubcli-archive-keyring.gpg
!echo "deb [arch=$(dpkg --print-architecture) signed-by=/usr/share/keyrings/githubcli-archive-keyring.gpg] https://cli.github.com/packages stable main" | tee /etc/apt/sources.list.d/github-cli.list > /dev/null
!apt update
!apt install gh -y

### 4.2 Descarga de Scripts desde GitHub

In [ ]:
GITHUB_REPO = "https://github.com/snaraya1/FASE_1.git"
RUTA_REPO = "/content/FASE_1"

if os.path.exists(RUTA_REPO):
    shutil.rmtree(RUTA_REPO)

!git clone {GITHUB_REPO} {RUTA_REPO}
print(f"Repositorio clonado en: {RUTA_REPO}")

### 4.3 Configuración de Rutas para Análisis

In [ ]:
RUTA_SCRIPTS = f"{RUTA_REPO}/scripts"
RUTA_SECUENCIAS = f"{RUTA_REPO}/secuencias"
RUTA_ALINEAMIENTOS = "/content/alineamientos"
RUTA_RESULTADOS = "/content/resultados"

os.makedirs(RUTA_ALINEAMIENTOS, exist_ok=True)
os.makedirs(RUTA_RESULTADOS, exist_ok=True)

sys.path.insert(0, RUTA_SCRIPTS)

print(f"Scripts disponibles en: {RUTA_SCRIPTS}")
print(f"Archivos de script: {os.listdir(RUTA_SCRIPTS)}")

### 4.4 Procesamiento de Secuencias por Gen

In [ ]:
GENES_ANALISIS = [
    "OG0000029_OipA", "OG0000047_BabA", "OG0000053_UreB", "OG0000123_VacA",
    "OG0000366_NapA", "OG0000677_HpaA", "OG0001247_HomA", "OG0001326_CagA"
]

for gen in GENES_ANALISIS:
    archivo_fasta = f"{RUTA_SECUENCIAS}/{gen}.fasta"
    
    if not os.path.exists(archivo_fasta):
        print(f"Advertencia: No se encontró {archivo_fasta}")
        continue
    
    archivo_alineamiento = f"{RUTA_ALINEAMIENTOS}/{gen}_aligned.fasta"
    
    print(f"\nProcesando {gen}...")
    !mafft --auto {archivo_fasta} > {archivo_alineamiento}
    
    print(f"Alineamiento guardado en: {archivo_alineamiento}")

### 4.5 Cálculo de Estadísticos de Diversidad

In [ ]:
from calcular_diversidad import calcular_estadisticos_diversidad

resultados_diversidad = []

for gen in GENES_ANALISIS:
    archivo_alineamiento = f"{RUTA_ALINEAMIENTOS}/{gen}_aligned.fasta"
    
    if not os.path.exists(archivo_alineamiento):
        continue
    
    print(f"\nCalculando diversidad para {gen}...")
    stats = calcular_estadisticos_diversidad(archivo_alineamiento)
    
    stats['Gen'] = gen
    resultados_diversidad.append(stats)

df_diversidad = pd.DataFrame(resultados_diversidad)
archivo_salida = f"{RUTA_RESULTADOS}/diversidad_genetica.csv"
df_diversidad.to_csv(archivo_salida, index=False)

print(f"\nResultados guardados en: {archivo_salida}")
print(df_diversidad)

## 5. Análisis de Diferenciación Genética (Fst)

### 5.1 Clasificación de Poblaciones

In [ ]:
def clasificar_poblacion(nombre_secuencia):
    """Clasifica secuencias en población Perú o Mundo."""
    if "Peru" in nombre_secuencia or "GCA_0167" in nombre_secuencia:
        return "Peru"
    else:
        return "Mundo"

def preparar_datos_fst(archivo_alineamiento):
    """Prepara datos para análisis de Fst."""
    secuencias = list(SeqIO.parse(archivo_alineamiento, "fasta"))
    
    poblaciones = {}
    for seq in secuencias:
        pop = clasificar_poblacion(seq.id)
        if pop not in poblaciones:
            poblaciones[pop] = []
        poblaciones[pop].append(str(seq.seq))
    
    return poblaciones

### 5.2 Cálculo de Fst por Sitio

In [ ]:
from calcular_fst import calcular_fst_por_sitio

resultados_fst = []

for gen in GENES_ANALISIS:
    archivo_alineamiento = f"{RUTA_ALINEAMIENTOS}/{gen}_aligned.fasta"
    
    if not os.path.exists(archivo_alineamiento):
        continue
    
    print(f"\nCalculando Fst para {gen}...")
    poblaciones = preparar_datos_fst(archivo_alineamiento)
    
    fst_por_sitio = calcular_fst_por_sitio(
        poblaciones['Peru'],
        poblaciones['Mundo']
    )
    
    for posicion, fst_valor in enumerate(fst_por_sitio, 1):
        resultados_fst.append({
            'Gen': gen,
            'Posicion': posicion,
            'Fst': fst_valor
        })

df_fst = pd.DataFrame(resultados_fst)
archivo_fst = f"{RUTA_RESULTADOS}/fst_por_sitio.csv"
df_fst.to_csv(archivo_fst, index=False)

print(f"\nResultados de Fst guardados en: {archivo_fst}")

## 6. Visualización: Manhattan Plot

### 6.1 Descarga de Datos para Visualización

In [ ]:
file_id_manhattan = "1JOXcCMqMo9wuKAp6jIE6Iym0A9XJ_TOX"
gdown.download(
    f"https://drive.google.com/uc?id={file_id_manhattan}",
    "manhattan_plot_fase1.csv",
    quiet=False
)

df_manhattan = pd.read_csv("manhattan_plot_fase1.csv")
print(f"Datos cargados: {len(df_manhattan)} filas")

### 6.2 Configuración de Parámetros del Plot

In [ ]:
info_genes = {
    "OG0001326_CagA": (0.12059, 0.43030),
    "OG0000047_BabA": (0.05554, 0.37169),
    "OG0000053_UreB": (0.05464, 0.17857),
    "OG0000677_HpaA": (0.04784, 0.24910),
    "OG0001247_HomA": (0.04578, 0.23266),
    "OG0000366_NapA": (0.03568, 0.17729),
    "OG0000029_OipA": (0.03427, 0.27024),
    "OG0000123_VacA": (0.01593, 0.20482),
}

ORDEN_GENES = [
    "OG0001326_CagA", "OG0000047_BabA", "OG0000053_UreB",
    "OG0000677_HpaA", "OG0001247_HomA", "OG0000366_NapA",
    "OG0000029_OipA", "OG0000123_VacA",
]

### 6.3 Construcción del Manhattan Plot

In [ ]:
GAP = 80
df_manhattan["x_global"] = np.nan
gene_spans = {}
cursor = 0

for gen in ORDEN_GENES:
    sub = df_manhattan[df_manhattan["Gen"] == gen].copy()
    if sub.empty:
        continue
    
    x0 = cursor
    x1 = cursor + sub["Posicion"].max()
    df_manhattan.loc[df_manhattan["Gen"] == gen, "x_global"] = sub["Posicion"] + cursor
    gene_spans[gen] = (x0, x1, (x0 + x1) / 2)
    cursor = x1 + GAP

GREY = "#aaaaaa"
RED = "#d62828"

fig, ax = plt.subplots(figsize=(14, 5))
fig.patch.set_facecolor("white")
ax.set_facecolor("white")

for gen in ORDEN_GENES:
    if gen not in gene_spans:
        continue

    sub = df_manhattan[df_manhattan["Gen"] == gen]
    x0, x1, mid = gene_spans[gen]
    fst_m, p99 = info_genes[gen]

    mask_grey = sub["Fst"] < p99
    ax.scatter(
        sub.loc[mask_grey, "x_global"],
        sub.loc[mask_grey, "Fst"],
        color=GREY, s=4, alpha=0.5, linewidths=0, zorder=2
    )

    mask_sig = ~mask_grey
    ax.scatter(
        sub.loc[mask_sig, "x_global"],
        sub.loc[mask_sig, "Fst"],
        color=RED, s=10, alpha=0.95, linewidths=0, zorder=3
    )

    ax.axvline(x0, color="#cccccc", lw=0.6, zorder=1)
    ax.axvline(x1, color="#cccccc", lw=0.6, zorder=1)

    short = gen.split("_")[1]
    ax.annotate(
        short,
        xy=(mid, 0),
        xycoords=("data", "axes fraction"),
        xytext=(0, -12),
        textcoords="offset points",
        ha="center", va="top",
        fontsize=10, fontweight="bold",
        color="black", rotation=0,
        annotation_clip=False
    )

ax.set_xlim(-80, cursor + 60)
ax.set_ylim(0, df_manhattan["Fst"].max() + 0.06)
ax.set_ylabel("Valores $F_{ST}$", fontsize=11, labelpad=6)
ax.set_xlabel("Sitio nucleotídico", fontsize=11, labelpad=25)
ax.set_title(
    "Análisis de $F_{ST}$ por Sitio Nucleotídico (Diferenciación Perú vs. Mundo)",
    fontsize=12, pad=10, loc="left"
)

ax.tick_params(axis="y", labelsize=9)
ax.tick_params(axis="x", bottom=False, labelbottom=False)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_color("#888888")
ax.spines["left"].set_color("#888888")
ax.yaxis.grid(True, color="#eeeeee", lw=0.6)

legend_elements = [
    mpatches.Patch(color=GREY, label="$F_{ST}$ < P99"),
    mpatches.Patch(color=RED, label="$F_{ST}$ ≥ P99"),
]
ax.legend(
    handles=legend_elements,
    loc="upper right",
    frameon=True,
    fontsize=9,
    edgecolor="#cccccc"
)

plt.savefig("manhattan_fst_final.png", dpi=200, bbox_inches="tight", facecolor="white")
plt.show()

print("Manhattan plot guardado como: manhattan_fst_final.png")